# Notebook 01 — Data Understanding & Initial EDA

**Project:** Customer Churn ETL & Analytics Pipeline  
**Student:** Divyesh Joshi | MC24097 | MCA-II Sem IV  
**Institute:** IIMS Chinchwad, Pune — Savitribai Phule Pune University  
**Guide:** Prof. Yugandhara Patil  

---

## Objective
This notebook covers **Phase 1** of the ETL pipeline:
- Load and inspect the raw IBM Telco Customer Churn dataset
- Understand the shape, columns, and data types
- Identify missing values and duplicates
- Explore the target variable distribution
- Generate initial statistical summaries

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

print('Libraries loaded successfully.')

ModuleNotFoundError: No module named 'pandas'

## 2. Load Dataset

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

FILE_PATH = '../data/raw/Telco-Customer-Churn.csv'
df = pd.read_csv(FILE_PATH)

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')

## 3. Dataset Shape & Column Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumn Names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

In [ ]:
# Data types
print('Data Types:\n')
print(df.dtypes)

## 4. First Look at the Data

In [ ]:
df.head(10)

In [ ]:
df.tail(5)

## 5. Statistical Summary

In [ ]:
print('Numeric Columns Summary:')
df.describe().round(2)

In [ ]:
print('Categorical Columns Summary:')
df.describe(include='object')

## 6. Missing Values Analysis

In [ ]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
null_df     = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
null_df     = null_df[null_df['Missing Count'] > 0]

if null_df.empty:
    print('No null values found in the dataset.')
else:
    print('Columns with missing values:')
    print(null_df)

In [ ]:
# TotalCharges has spaces — check it
print('TotalCharges sample values with spaces:')
print(df[df['TotalCharges'].str.strip() == ''][['customerID','tenure','TotalCharges']].head(10))
print(f'\nCount of blank TotalCharges: {(df["TotalCharges"].str.strip() == "").sum()}')

## 7. Duplicate Records

In [ ]:
dup_count = df.duplicated().sum()
print(f'Duplicate rows: {dup_count}')
if dup_count > 0:
    print(df[df.duplicated()])

## 8. Target Variable — Churn Distribution

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100

print('Churn Distribution:')
for label in churn_counts.index:
    print(f'  {label}: {churn_counts[label]:,} ({churn_pct[label]:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(x='Churn', data=df, ax=axes[0],
              palette=['#2a5298','#e74c3c'])
axes[0].set_title('Churn Count', fontweight='bold')
axes[0].bar_label(axes[0].containers[0])

# Pie chart
axes[1].pie(churn_counts, labels=churn_counts.index,
            autopct='%1.1f%%', colors=['#2a5298','#e74c3c'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Percentage', fontweight='bold')

plt.suptitle('Target Variable — Churn Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('\nObservation: Dataset is imbalanced — 73.5% No Churn vs 26.5% Churn.')

## 9. Numeric Feature Distributions

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for i, col in enumerate(numeric_cols):
    # Histogram
    axes[0, i].hist(df[col].dropna(), bins=30, color='#2a5298', edgecolor='white', alpha=0.8)
    axes[0, i].set_title(f'{col} — Distribution', fontweight='bold')
    axes[0, i].set_xlabel(col)
    axes[0, i].set_ylabel('Frequency')

    # Boxplot by Churn
    churned = df[df['Churn']=='Yes'][col].dropna()
    stayed  = df[df['Churn']=='No'][col].dropna()
    axes[1, i].boxplot([stayed, churned], labels=['No Churn','Churned'],
                        patch_artist=True,
                        boxprops=dict(facecolor='#d0e4ff'),
                        medianprops=dict(color='#e74c3c', linewidth=2))
    axes[1, i].set_title(f'{col} vs Churn', fontweight='bold')

plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Categorical Feature Overview

In [ ]:
cat_cols = ['gender','SeniorCitizen','Partner','Dependents','PhoneService',
            'InternetService','Contract','PaymentMethod']

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    val_counts = df[col].value_counts()
    axes[i].bar(val_counts.index.astype(str), val_counts.values,
                color=sns.color_palette('Set2', len(val_counts)))
    axes[i].set_title(col, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=15)
    for bar in axes[i].patches:
        axes[i].annotate(f'{int(bar.get_height()):,}',
                         (bar.get_x() + bar.get_width()/2, bar.get_height()),
                         ha='center', va='bottom', fontsize=9)

plt.suptitle('Categorical Feature Value Counts', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Correlation Matrix

In [ ]:
corr_df = df[['tenure','MonthlyCharges','TotalCharges']].copy()
corr_df['Churn_flag'] = (df['Churn'] == 'Yes').astype(int)

plt.figure(figsize=(7, 5))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix — Numeric Features', fontweight='bold')
plt.tight_layout()
plt.show()
print('Observation: tenure has a negative correlation with Churn — longer tenure = less likely to churn.')

## 12. Summary & Findings

| Finding | Detail |
|---|---|
| Dataset size | 7,043 rows × 21 columns |
| Target imbalance | 73.5% No Churn / 26.5% Churn |
| Missing values | TotalCharges has 11 blank entries (new customers with tenure=0) |
| Duplicates | None found |
| Key observation | Month-to-month customers, fiber optic users, and electronic check payers show higher churn |
| Numeric correlation | Tenure negatively correlated with churn; MonthlyCharges positively correlated |

**Next Step → Notebook 02: Data Cleaning**